# Traffic Demand Prediction - Model Training
CatBoost + LightGBM + XGBoost + Ensemble

In [ ]:
!pip install catboost lightgbm xgboost -q


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.preprocessing import LabelEncoder
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import joblib

train = pd.read_csv(r'/mnt/data/traffic_data/dataset/train.csv')
test = pd.read_csv(r'/mnt/data/traffic_data/dataset/test.csv')

print(train.shape)
print(test.shape)
train.head()


## Feature Engineering

In [ ]:
train['timestamp']=pd.to_datetime(train['timestamp'])
test['timestamp']=pd.to_datetime(test['timestamp'])

for df in [train,test]:
    df['hour']=df['timestamp'].dt.hour
    df['month']=df['timestamp'].dt.month
    df['dayofweek']=df['timestamp'].dt.dayofweek
    df['is_weekend']=df['dayofweek'].isin([5,6]).astype(int)
    df.drop('timestamp',axis=1,inplace=True)


## Encoding

In [ ]:
cat_cols=train.select_dtypes(include='object').columns

encoders={}
for col in cat_cols:
    le=LabelEncoder()
    combined=pd.concat([train[col],test[col]],axis=0).astype(str)
    le.fit(combined)

    train[col]=le.transform(train[col].astype(str))
    test[col]=le.transform(test[col].astype(str))

    encoders[col]=le


## Train Validation Split

In [ ]:
X=train.drop(['demand'],axis=1)
y=train['demand']

X_train,X_val,y_train,y_val=train_test_split(
    X,y,test_size=0.2,random_state=42
)

print(X_train.shape,X_val.shape)


## CatBoost Model

In [ ]:
cat_model=CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    loss_function='RMSE',
    eval_metric='R2',
    verbose=200
)

cat_model.fit(X_train,y_train,
              eval_set=(X_val,y_val),
              use_best_model=True)

cat_pred=cat_model.predict(X_val)

cat_r2=r2_score(y_val,cat_pred)
print("CatBoost R2:",cat_r2)


## LightGBM Model

In [ ]:
lgb_model=LGBMRegressor(
    n_estimators=1200,
    learning_rate=0.03,
    max_depth=10,
    random_state=42
)

lgb_model.fit(X_train,y_train)

lgb_pred=lgb_model.predict(X_val)

lgb_r2=r2_score(y_val,lgb_pred)
print("LightGBM R2:",lgb_r2)


## XGBoost Model

In [ ]:
xgb_model=XGBRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb_model.fit(X_train,y_train)

xgb_pred=xgb_model.predict(X_val)

xgb_r2=r2_score(y_val,xgb_pred)
print("XGBoost R2:",xgb_r2)


## Ensemble Model

In [ ]:
ensemble_pred=(
    0.4*cat_pred+
    0.3*lgb_pred+
    0.3*xgb_pred
)

ensemble_r2=r2_score(y_val,ensemble_pred)

print("Ensemble R2:",ensemble_r2)


## Best Model Selection

In [ ]:
scores={
    'CatBoost':cat_r2,
    'LightGBM':lgb_r2,
    'XGBoost':xgb_r2,
    'Ensemble':ensemble_r2
}

best=max(scores,key=scores.get)

print(scores)
print("Best Model:",best)


## Train on Full Data and Predict

In [ ]:
cat_model.fit(X,y)

test_pred_cat=cat_model.predict(test)

lgb_model.fit(X,y)
test_pred_lgb=lgb_model.predict(test)

xgb_model.fit(X,y)
test_pred_xgb=xgb_model.predict(test)

final_pred=(
    0.4*test_pred_cat+
    0.3*test_pred_lgb+
    0.3*test_pred_xgb
)


## Submission File

In [ ]:
submission=pd.DataFrame({
    'Index':test['Index'],
    'demand':final_pred
})

submission.to_csv('submission.csv',index=False)

submission.head()


## Save Models

In [ ]:
joblib.dump(cat_model,'catboost_model.pkl')
joblib.dump(lgb_model,'lightgbm_model.pkl')
joblib.dump(xgb_model,'xgboost_model.pkl')

print("Models Saved")
